# v7.1 — Stage-B reject class + gate threshold sweep (full release)

v7 (`final_model_v7_full.ipynb`) showed the macro-F1 deficit at the real base rate is **benign
leakage**: 8,370 benign flows pass the gate and Stage B is *forced* to call every one an attack
(6,327 land in Fuzzers). Two levers, both aimed at that mechanism:

1. **Reject class** — Stage B becomes 10-class: 9 families + "Benign-that-fooled-the-gate",
   trained on the ~34k OOF gate false positives. A leaked flow can now be sent back to Benign.
2. **Gate threshold sweep** — re-tested here because the curated-file null (`two_stage_gate_weighting`)
   was measured at a 4:1 base rate and without a reject path; both conditions changed.

**Protocol:** same data prep, splits and seeds as v7 (asserted); reuses `cache_oof_v7_full.npz`;
two new caches (gate OOF *probabilities*, 10-class reject-member OOF). Selection: SELV once across
{baseline, reject} × 21 variants × thresholds; **test touched once** for the selected config, plus
the pre-registered paired comparison reject-vs-baseline at t=0.5.

**Deviations (stated):** reject-set membership comes from OOF gate *hard* predictions (mild fold
reuse — a row's OOF gate model saw other folds' val rows); the reject set is defined at t=0.5 even
for other thresholds in the sweep.

**Runtime ≈ 40–50 min first run** (reject-member cross-fit ~20–25 min); cached thereafter.
Requires `cache_oof_v7_full.npz` from v7.

In [1]:
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

CFM = Path("Resource/CICFlowMeter_out.csv")
RS, EPS = 42, 1e-6
CAT = {"Benign": 0, "Analysis": 1, "Backdoor": 2, "DoS": 3, "Exploits": 4,
       "Fuzzers": 5, "Generic": 6, "Reconnaissance": 7, "Shellcode": 8, "Worms": 9}
NAMES = list(CAT.keys())

## 1–2. Data prep — byte-identical to v7 (splits asserted at the end)

In [2]:
t0 = time.time()
hdr = pd.read_csv(CFM, nrows=0).columns.tolist()
FEAT76 = hdr[7:-1]
dtypes = {c: np.float32 for c in FEAT76}
dtypes["Dst Port"] = np.int32; dtypes["Protocol"] = np.int16
df = pd.read_csv(CFM, usecols=["Dst Port", "Protocol"] + FEAT76 + ["Label"], dtype=dtypes)
df = df.drop_duplicates().reset_index(drop=True)
y = df["Label"].map(CAT).astype(int).values

Xraw = df[FEAT76]
nun = Xraw.nunique(); zero = nun[nun <= 1].index.tolist()
FULL = Xraw.drop(columns=zero)
corr = FULL.corr(); upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE = FULL.drop(columns=red)

def safe(a, b):
    return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)

F = FULL
g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)
fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
tot_pk, tot_by = fpk + bpk, fby + bby
d = {}
d["r_pkt_fwd_bwd"] = safe(fpk, bpk); d["r_byte_fwd_bwd"] = safe(fby, bby)
d["asym_pkt"] = safe(fpk - bpk, tot_pk); d["asym_byte"] = safe(fby - bby, tot_by)
d["bpp_fwd"] = safe(fby, fpk); d["bpp_bwd"] = safe(bby, bpk); d["bpp_flow"] = safe(tot_by, tot_pk)
d["r_bpp"] = safe(d["bpp_fwd"], d["bpp_bwd"])
d["cv_flow_iat"] = safe(g("Flow IAT Std"), g("Flow IAT Mean"))
d["cv_fwd_iat"] = safe(g("Fwd IAT Std"), g("Fwd IAT Mean"))
d["cv_bwd_iat"] = safe(g("Bwd IAT Std"), g("Bwd IAT Mean"))
d["iat_span_rel"] = safe(g("Flow IAT Max") - g("Flow IAT Min"), g("Flow IAT Mean"))
d["cv_fwd_len"] = safe(g("Fwd Packet Length Std"), g("Fwd Packet Length Mean"))
d["cv_bwd_len"] = safe(g("Bwd Packet Length Std"), g("Bwd Packet Length Mean"))
d["cv_pkt_len"] = safe(g("Packet Length Std"), g("Packet Length Mean"))
d["fwd_len_span_rel"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"), g("Fwd Packet Length Mean"))
d["bwd_len_span_rel"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"), g("Bwd Packet Length Mean"))
for flag in ("FIN", "SYN", "RST", "PSH", "ACK"):
    d["frac_" + flag] = safe(g(flag + " Flag Count"), tot_pk)
d["r_syn_fin"] = safe(g("SYN Flag Count"), g("FIN Flag Count"))
d["hdr_per_pkt_fwd"] = safe(g("Fwd Header Length"), fpk)
d["hdr_per_pkt_bwd"] = safe(g("Bwd Header Length"), bpk)
d["payload_share_fwd"] = safe(fby - g("Fwd Header Length"), fby)
dur = g("Flow Duration")
d["active_share"] = safe(g("Active Mean"), dur); d["idle_share"] = safe(g("Idle Mean"), dur)
d["r_active_idle"] = safe(g("Active Mean"), g("Idle Mean")); d["dur_per_pkt"] = safe(dur, tot_pk)
d["r_init_win"] = safe(g("FWD Init Win Bytes"), g("Bwd Init Win Bytes"))
DER = pd.DataFrame(d, index=F.index).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)
dz = DER.nunique(); DER = DER.drop(columns=dz[dz <= 1].index.tolist())

port = df["Dst Port"].values
proto = df["Protocol"].values
SVC_PORTS = [(20, "svc_ftpdata"), (21, "svc_ftp"), (22, "svc_ssh"), (23, "svc_telnet"),
             (25, "svc_smtp"), (53, "svc_dns"), (80, "svc_http"), (110, "svc_pop3"),
             (111, "svc_rpc"), (139, "svc_netbios"), (143, "svc_imap"), (179, "svc_bgp"),
             (443, "svc_https"), (445, "svc_smb"), (514, "svc_syslog"), (520, "svc_rip"),
             (1723, "svc_pptp"), (5190, "svc_aim"), (6881, "svc_bt"), (8080, "svc_httpalt")]
KNOWN = np.array([p for p, _ in SVC_PORTS])
s = {nm: (port == p).astype(np.float32) for p, nm in SVC_PORTS}
s["svc_other_low"] = ((port < 1024) & ~np.isin(port, KNOWN)).astype(np.float32)
s["svc_registered"] = ((port >= 1024) & (port < 49152) & ~np.isin(port, KNOWN)).astype(np.float32)
s["svc_ephemeral"] = (port >= 49152).astype(np.float32)
s["proto_tcp"] = (proto == 6).astype(np.float32)
s["proto_udp"] = (proto == 17).astype(np.float32)
s["dst_port_num"] = port.astype(np.float32)
SVC = pd.DataFrame(s, index=df.index)
sz = SVC.nunique(); SVC = SVC.drop(columns=sz[sz <= 1].index.tolist())
X = pd.concat([BASE.astype(np.float32), DER, SVC], axis=1)
COLS = list(X.columns)
del d, Xraw, FULL, BASE, DER, SVC, corr, upper, s, df, F
del fpk, bpk, fby, bby, tot_pk, tot_by, dur; gc.collect()

idx_all = np.arange(len(y))
i_trf, i_test = train_test_split(idx_all, test_size=0.2, random_state=RS, stratify=y)
X_trf = X.iloc[i_trf].reset_index(drop=True)
X_test = X.iloc[i_test].reset_index(drop=True)
y_arr = y[i_trf]; y_te = y[i_test]
del X; gc.collect()

attack_labels = sorted(l for l in np.unique(y_arr) if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
NC = len(attack_labels)                    # 9 families
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values
att_te = y_te != 0

comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
in_comb = np.zeros(len(y_arr), bool); in_comb[comb_idx] = True

# assert we are on v7's exact splits
assert len(y_arr) == 2085442 and len(y_te) == 521361, (len(y_arr), len(y_te))
assert att_trf.sum() == 65191 and X_trf.shape[1] == 99
print("prep done in %.0fm -- splits match v7" % ((time.time() - t0) / 60))

prep done in 1m -- splits match v7


## 3. Load v7 OOF cache + build the two new caches

3a: v7's 9-class member OOF + hard gate OOF (required).
3b: **gate OOF probabilities** (5 gate refits — same folds, `predict_proba` kept this time).
3c: **10-class reject-member OOF** — reject set = Benign rows whose OOF gate said "attack".

In [3]:
CACHE7 = Path("cache_oof_v7_full.npz")
assert CACHE7.exists(), "run final_model_v7_full.ipynb first"
z = np.load(CACHE7)
M = 7
OOF9 = [z[f"m{m}"] for m in range(M)]; oof_gate = z["gate"]
assert len(oof_gate) == len(y_arr)
print("v7 OOF loaded:", CACHE7)

REJECT = (y_arr == 0) & (oof_gate == 1)          # benign that fooled the OOF gate
NCR = NC + 1; IREJ = NC                          # reject = class index 9
FAMR = [NAMES[l] for l in attack_labels] + ["REJECT->Benign"]
yr10 = np.where(REJECT, IREJ, fam_trf)           # -1 for benign non-FP rows
print("reject set: %d benign OOF-gate FPs (%.3f%% of benign train+val)"
      % (REJECT.sum(), 100 * REJECT.sum() / (y_arr == 0).sum()))

v7 OOF loaded: cache_oof_v7_full.npz
reject set: 34101 benign OOF-gate FPs (1.688% of benign train+val)


In [4]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw); return XGBClassifier(**p)

def wc_alpha(n, a):
    w = (n.sum() / n) ** a; return w / w.mean()

def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()

SPECS = ([(f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0)]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s_[0] for s_ in SPECS]

def fit_member(kind, param, Xa, ya, ncls):
    n = np.bincount(np.asarray(ya), minlength=ncls)
    if kind == "smote":
        strat = {c: param for c in range(ncls) if 0 < n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if kind == "alpha":  m = make_xgb()
    elif kind == "rf":   m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":   m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    g_ = make_xgb(); g_.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
    return g_

folds = list(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr))

In [5]:
# --- 3b: gate OOF probabilities ---
CACHE_GP = Path("cache_oof_v71_gateproba.npz")
if CACHE_GP.exists():
    p_gate = np.load(CACHE_GP)["p"]
    print("loaded", CACHE_GP)
else:
    p_gate = np.zeros(len(y_arr), np.float32)
    t0 = time.time()
    for f_, (tr, va) in enumerate(folds):
        gm = fit_gate(X_trf.iloc[tr], y_arr[tr])
        p_gate[va] = gm.predict_proba(X_trf.iloc[va])[:, 1].astype(np.float32)
        del gm; gc.collect()
        print("  fold %d gate proba (%.0fs)" % (f_ + 1, time.time() - t0))
    np.savez_compressed(CACHE_GP, p=p_gate)
agree = float(((p_gate > 0.5).astype(np.int8) == oof_gate).mean())
print("agreement with v7 cached hard gate at t=0.5: %.4f%%" % (100 * agree))

  fold 1 gate proba (30s)
  fold 2 gate proba (59s)
  fold 3 gate proba (89s)
  fold 4 gate proba (118s)
  fold 5 gate proba (146s)
agreement with v7 cached hard gate at t=0.5: 100.0000%


In [6]:
# --- 3c: 10-class reject-member OOF ---
CACHE_R = Path("cache_oof_v71_reject.npz")
if CACHE_R.exists():
    zr = np.load(CACHE_R)
    OOF10 = [zr[f"m{m}"] for m in range(M)]
    print("loaded", CACHE_R)
else:
    OOF10 = [np.zeros((len(y_arr), NCR), np.float32) for _ in range(M)]
    t0 = time.time()
    for f_, (tr, va) in enumerate(folds):
        tf = time.time()
        m_use = yr10[tr] >= 0                      # attacks + reject rows of this fold-train
        Xa = X_trf.iloc[tr[m_use]].reset_index(drop=True)
        ya = pd.Series(yr10[tr[m_use]])
        for mi, (nm, (kind, param)) in enumerate(SPECS):
            mdl = fit_member(kind, param, Xa, ya, NCR)
            OOF10[mi][va] = mdl.predict_proba(X_trf.iloc[va]).astype(np.float32)
            del mdl; gc.collect()
        print("  fold %d/5 done (%.0fs, elapsed %.0fm, train rows %d)"
              % (f_ + 1, time.time() - tf, (time.time() - t0) / 60, len(ya)))
    np.savez_compressed(CACHE_R, **{f"m{m}": OOF10[m] for m in range(M)})
    print("reject cross-fit complete in %.0fm" % ((time.time() - t0) / 60))

  fold 1/5 done (140s, elapsed 2m, train rows 79238)
  fold 2/5 done (141s, elapsed 5m, train rows 79572)
  fold 3/5 done (139s, elapsed 7m, train rows 79461)
  fold 4/5 done (139s, elapsed 9m, train rows 79470)
  fold 5/5 done (140s, elapsed 12m, train rows 79427)
reject cross-fit complete in 12m


## 4. Combiners — same machinery, parameterised by arm

Arm state = (OOF, reference labels incl. reject, class count, index→label map). `W` is fitted on the
COMB half; the reject arm's reference set also contains the COMB-half reject rows, so the combiner
learns when to trust the reject vote.

In [7]:
GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)

def fuse_static(probas, Wclass, M_=7):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M_):
        S += probas[m] * Wclass[m][None, :]
    return S.argmax(1)

def fuse_dyn(probas, Wd, p=1.0, Wclass=None, M_=7):
    Wp = np.power(np.clip(Wd, 1e-6, None), p); Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M_):
        coef = Wp[:, m:m + 1] if Wclass is None else Wp[:, m:m + 1] * Wclass[m][None, :]
        S += coef * probas[m]
    return S.argmax(1)

def fit_W(probas, ytrue, ncls, passes=3):
    W = np.ones((M, ncls)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(ncls):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s_ = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s_ > best + 1e-7: best, cur = s_, gg
                W[m_, c] = cur
    return W, best

prof = lambda ps: np.hstack(ps).astype(np.float32)

def build_arm(OOF, ref_mask, yref_all, ncls, lab_of_idx, name):
    ref = np.where(ref_mask)[0]
    y_ref = yref_all[ref]
    st = dict(name=name, OOF=OOF, ncls=ncls, LAB=np.asarray(lab_of_idx), ref=ref, y_ref=y_ref)
    st["CORRECT"] = np.stack([(OOF[m][ref].argmax(1) == y_ref) for m in range(M)])
    st["WC"] = wc_alpha(np.bincount(y_ref, minlength=ncls), 0.25)
    st["NN"] = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
        prof([OOF[m][ref] for m in range(M)]))
    t0 = time.time()
    st["W"], sb = fit_W([OOF[m][ref] for m in range(M)], y_ref, ncls)
    print("%s: ref %d rows | W fitted %.0fs (Stage-B mF1 %.4f)"
          % (name, len(ref), time.time() - t0, sb))
    return st

def comp_ola(st, idx, k):
    nb = idx[:, :k]; wn = st["WC"][st["y_ref"][nb]]; den = wn.sum(1)
    Wd = np.empty((nb.shape[0], M))
    for m in range(M):
        Wd[:, m] = (st["CORRECT"][m][nb] * wn).sum(1) / den
    return Wd

def comp_lca(st, probas, idx, k):
    nb = idx[:, :k]; yn = st["y_ref"][nb]; Wd = np.empty((nb.shape[0], M))
    for m in range(M):
        pq = probas[m].argmax(1); mask = yn == pq[:, None]; cnt = mask.sum(1)
        Wd[:, m] = np.where(cnt > 0, (st["CORRECT"][m][nb] & mask).sum(1) / np.maximum(cnt, 1),
                            st["CORRECT"][m][nb].mean(1))
    return Wd

def score_e2e(st, fam_g, ye, gmask, tag):
    yp = np.zeros(len(ye), int)
    if len(fam_g): yp[gmask] = st["LAB"][fam_g]
    return dict(variant=tag,
                e2e_mF1=round(f1_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mP=round(precision_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mR=round(recall_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye).mean()), 4))

def variants(st, P_g, idx):
    eqW = np.ones((M, st["ncls"])) / M
    out = {"STATIC: per-class W": fuse_static(P_g, st["W"]),
           "STATIC: equal average": fuse_static(P_g, eqW)}
    for m in range(M):
        out["single: " + MODELS[m]] = P_g[m].argmax(1)
    for k in (30, 100):
        for p in (1, 3, 8):
            out[f"DYN OLA-cw k={k} p={p}"] = fuse_dyn(P_g, comp_ola(st, idx, k), p=p)
    out["DYN LCA k=30"] = fuse_dyn(P_g, comp_lca(st, P_g, idx, 30))
    for k in (30, 100):
        for p in (1, 3):
            out[f"HYBRID W x OLA-cw k={k} p={p}"] = fuse_dyn(P_g, comp_ola(st, idx, k), p=p,
                                                             Wclass=st["W"])
    out["HYBRID W x LCA k=30 p=1"] = fuse_dyn(P_g, comp_lca(st, P_g, idx, 30), p=1, Wclass=st["W"])
    return out

LAB9 = np.array(attack_labels)                    # 0..8 -> labels 1..9
LAB10 = np.array(attack_labels + [0])             # index 9 -> Benign
ARM_B = build_arm(OOF9, in_comb & att_trf, fam_trf, NC, LAB9, "baseline")
ARM_R = build_arm(OOF10, in_comb & (yr10 >= 0), yr10, NCR, LAB10, "reject")

baseline: ref 32596 rows | W fitted 15s (Stage-B mF1 0.7626)
reject: ref 49541 rows | W fitted 26s (Stage-B mF1 0.6604)


## 5. SELV: 21 variants per arm at t=0.5, then the threshold sweep on each winner

In [8]:
def selv_eval(st, t):
    gmask = p_gate[selv_idx] > t
    gpos = selv_idx[gmask]
    P = [st["OOF"][m][gpos] for m in range(M)]
    _, idx = st["NN"].kneighbors(prof(P))
    return P, idx, gmask

rows = []
WINNER = {}
for st in (ARM_B, ARM_R):
    P, idx, gmask = selv_eval(st, 0.5)
    tab = pd.DataFrame([score_e2e(st, fg, y_arr[selv_idx], gmask, v)
                        for v, fg in variants(st, P, idx).items()]
                       ).sort_values("e2e_mF1", ascending=False).reset_index(drop=True)
    print("\n--- SELV %s (t=0.5) ---" % st["name"])
    print(tab.head(8).to_string(index=False))
    WINNER[st["name"]] = tab.iloc[0]["variant"]
    rows.append(dict(arm=st["name"], variant=tab.iloc[0]["variant"], t=0.5,
                     selv_mF1=tab.iloc[0]["e2e_mF1"]))
    print("winner: %s (%.4f)" % (tab.iloc[0]["variant"], tab.iloc[0]["e2e_mF1"]))
del P, idx; gc.collect()


--- SELV baseline (t=0.5) ---
                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
        DYN OLA-cw k=30 p=1   0.6449  0.6407  0.7165   0.9793
        DYN OLA-cw k=30 p=3   0.6448  0.6372  0.7215   0.9794
       DYN OLA-cw k=100 p=3   0.6448  0.6402  0.7186   0.9793
HYBRID W x OLA-cw k=100 p=3   0.6445  0.6259  0.7382   0.9794
       DYN OLA-cw k=100 p=8   0.6445  0.6360  0.7246   0.9794
 HYBRID W x OLA-cw k=30 p=3   0.6442  0.6246  0.7389   0.9794
               DYN LCA k=30   0.6440  0.6462  0.7098   0.9793
 HYBRID W x OLA-cw k=30 p=1   0.6439  0.6248  0.7375   0.9793
winner: DYN OLA-cw k=30 p=1 (0.6449)

--- SELV reject (t=0.5) ---
                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
 HYBRID W x OLA-cw k=30 p=3   0.6695  0.7046  0.6560   0.9870
    HYBRID W x LCA k=30 p=1   0.6682  0.7062  0.6528   0.9868
 HYBRID W x OLA-cw k=30 p=1   0.6677  0.7039  0.6540   0.9869
HYBRID W x OLA-cw k=100 p=3   0.6674  0.7023  0.6547   0.9869
HYBRID W x OLA-cw k=100 p=1   0.667

0

In [9]:
def one_variant(st, vname, P_g, idx):
    return variants(st, P_g, idx)[vname]   # simple; recomputes the family, fine at this scale

THRESH = (0.3, 0.4, 0.5, 0.6, 0.7)
sweep = []
for st in (ARM_B, ARM_R):
    vn = WINNER[st["name"]]
    for t in THRESH:
        P, idx, gmask = selv_eval(st, t)
        r = score_e2e(st, one_variant(st, vn, P, idx), y_arr[selv_idx], gmask,
                      "%s | %s | t=%.1f" % (st["name"], vn, t))
        r.update(arm=st["name"], vname=vn, t=t)
        sweep.append(r)
        del P, idx; gc.collect()
sw = pd.DataFrame(sweep).sort_values("e2e_mF1", ascending=False).reset_index(drop=True)
print(sw[["variant", "e2e_mF1", "e2e_mP", "e2e_mR", "e2e_acc"]].to_string(index=False))
FINAL_ARM, FINAL_V, FINAL_T = sw.iloc[0]["arm"], sw.iloc[0]["vname"], float(sw.iloc[0]["t"])
print("\nFINAL config (selected on SELV): arm=%s  variant=%s  t=%.1f" % (FINAL_ARM, FINAL_V, FINAL_T))

                                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
reject | HYBRID W x OLA-cw k=30 p=3 | t=0.7   0.6711  0.7080  0.6525   0.9871
reject | HYBRID W x OLA-cw k=30 p=3 | t=0.6   0.6709  0.7071  0.6553   0.9870
reject | HYBRID W x OLA-cw k=30 p=3 | t=0.5   0.6695  0.7046  0.6560   0.9870
reject | HYBRID W x OLA-cw k=30 p=3 | t=0.4   0.6687  0.7032  0.6566   0.9869
reject | HYBRID W x OLA-cw k=30 p=3 | t=0.3   0.6679  0.7000  0.6586   0.9869
     baseline | DYN OLA-cw k=30 p=1 | t=0.7   0.6570  0.6610  0.6988   0.9811
     baseline | DYN OLA-cw k=30 p=1 | t=0.6   0.6522  0.6517  0.7097   0.9801
     baseline | DYN OLA-cw k=30 p=1 | t=0.5   0.6449  0.6407  0.7165   0.9793
     baseline | DYN OLA-cw k=30 p=1 | t=0.4   0.6395  0.6328  0.7246   0.9787
     baseline | DYN OLA-cw k=30 p=1 | t=0.3   0.6328  0.6235  0.7298   0.9779

FINAL config (selected on SELV): arm=reject  variant=HYBRID W x OLA-cw k=30 p=3  t=0.7


## 6. Refit on train+val, TEST ONCE

Test is scored for exactly three pre-declared things: the v7 baseline at t=0.5 (reference), the
reject arm at t=0.5 (the pre-registered paired comparison), and the SELV-selected FINAL config.

In [10]:
t0 = time.time()
gate_f = fit_gate(X_trf, y_arr)
Xa9 = X_trf[att_trf].reset_index(drop=True)
FINAL9 = [fit_member(k, p, Xa9, pd.Series(fam_trf[att_trf]), NC) for _, (k, p) in SPECS]
m10 = yr10 >= 0
Xa10 = X_trf[m10].reset_index(drop=True)
FINAL10 = [fit_member(k, p, Xa10, pd.Series(yr10[m10]), NCR) for _, (k, p) in SPECS]
print("gate + both pools refit in %.0fm" % ((time.time() - t0) / 60))
p_test = gate_f.predict_proba(X_test)[:, 1].astype(np.float32)

def test_pred(st, POOL, vname, t):
    gmask = p_test > t
    gpos = np.where(gmask)[0]
    P = [m.predict_proba(X_test.iloc[gpos]).astype(np.float32) for m in POOL]
    _, idx = st["NN"].kneighbors(prof(P))
    fam_g = one_variant(st, vname, P, idx)
    yp = np.zeros(len(y_te), int)
    yp[gmask] = st["LAB"][fam_g]
    return yp

def summarize(tag, yp):
    print("%-34s mF1 %.4f  mP %.4f  mR %.4f  acc %.4f"
          % (tag, f1_score(y_te, yp, average="macro", zero_division=0),
             precision_score(y_te, yp, average="macro", zero_division=0),
             recall_score(y_te, yp, average="macro", zero_division=0),
             float((yp == y_te).mean())))

yp_base = test_pred(ARM_B, FINAL9, WINNER["baseline"], 0.5)
yp_rej  = test_pred(ARM_R, FINAL10, WINNER["reject"], 0.5)
if (FINAL_ARM, FINAL_T) == ("reject", 0.5):
    yp_fin = yp_rej
elif (FINAL_ARM, FINAL_T) == ("baseline", 0.5):
    yp_fin = yp_base
else:
    st_f = ARM_R if FINAL_ARM == "reject" else ARM_B
    yp_fin = test_pred(st_f, FINAL10 if FINAL_ARM == "reject" else FINAL9, FINAL_V, FINAL_T)

print("--- TEST ---")
summarize("baseline @0.5 (v7 reference)", yp_base)
summarize("reject   @0.5 (paired comparison)", yp_rej)
summarize("FINAL: %s %s t=%.1f" % (FINAL_ARM, FINAL_V, FINAL_T), yp_fin)
print("\nv7 notebook reference: 0.6601 / 0.9799 (its own refit; expect ~+/-0.001 jitter)")

gate + both pools refit in 5m
--- TEST ---
baseline @0.5 (v7 reference)       mF1 0.6601  mP 0.6568  mR 0.7329  acc 0.9799
reject   @0.5 (paired comparison)  mF1 0.6843  mP 0.7164  mR 0.6744  acc 0.9875
FINAL: reject HYBRID W x OLA-cw k=30 p=3 t=0.7 mF1 0.6842  mP 0.7178  mR 0.6691  acc 0.9875

v7 notebook reference: 0.6601 / 0.9799 (its own refit; expect ~+/-0.001 jitter)


In [11]:
print(classification_report(y_te, yp_fin, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_te, yp_fin, labels=list(range(10))),
                   index=NAMES, columns=NAMES).to_string())

gmask_f = p_test > FINAL_T
gr = recall_score((y_te != 0).astype(int), gmask_f.astype(int))
gp = precision_score((y_te != 0).astype(int), gmask_f.astype(int))
n_leak = int(((y_te == 0) & (yp_fin != 0)).sum())
print("\ngate@%.1f: attack recall %.4f / precision %.4f" % (FINAL_T, gr, gp))
print("benign flows FINALLY misclassified as attack: %d (v7: 8,370)" % n_leak)
if FINAL_ARM == "reject":
    n_rej = int((gmask_f & (yp_fin == 0)).sum())
    n_rej_ok = int((gmask_f & (yp_fin == 0) & (y_te == 0)).sum())
    n_rej_bad = n_rej - n_rej_ok
    print("gate-positive rows sent back to Benign by the reject class: %d (%d correct, %d were real attacks)"
          % (n_rej, n_rej_ok, n_rej_bad))

                precision    recall  f1-score   support

        Benign      0.995     0.996     0.995    505063
      Analysis      0.090     0.260     0.134        77
      Backdoor      0.797     0.567     0.662        90
           DoS      0.656     0.555     0.601       884
      Exploits      0.782     0.863     0.821      6185
       Fuzzers      0.676     0.576     0.622      5287
       Generic      0.873     0.882     0.878       899
Reconnaissance      0.882     0.774     0.825      2407
     Shellcode      0.859     0.626     0.725       420
         Worms      0.569     0.592     0.580        49

      accuracy                          0.988    521361
     macro avg      0.718     0.669     0.684    521361
  weighted avg      0.987     0.988     0.987    521361

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign          502970       166         4   44       360     1386       33              77         19 

### 6.1 Paired bootstrap (reject vs baseline, both at t=0.5) and per-class deltas

In [12]:
def macro_f1_from_counts(cm):
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    return (2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)).mean()

code_a = y_te * 10 + yp_base
code_b = y_te * 10 + yp_rej
rng = np.random.RandomState(RS)
cls_idx = [np.where(y_te == c)[0] for c in range(10)]
diffs = np.empty(2000); f1b = np.empty(2000)
for it in range(2000):
    bidx = np.concatenate([ci[rng.randint(0, len(ci), len(ci))] for ci in cls_idx])
    cma = np.bincount(code_a[bidx], minlength=100).reshape(10, 10)
    cmb = np.bincount(code_b[bidx], minlength=100).reshape(10, 10)
    fb = macro_f1_from_counts(cmb); f1b[it] = fb
    diffs[it] = fb - macro_f1_from_counts(cma)
lo, hi = np.percentile(diffs, [2.5, 97.5])
p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
mf_a = f1_score(y_te, yp_base, average="macro", zero_division=0)
mf_b = f1_score(y_te, yp_rej, average="macro", zero_division=0)
blo, bhi = np.percentile(f1b, [2.5, 97.5])
print("paired delta (reject - baseline) @t=0.5: %+.4f  95%% CI [%+.4f, %+.4f]  p ~ %.3f"
      % (mf_b - mf_a, lo, hi, min(1.0, p_two)))
print("reject marginal: %.4f [%.4f, %.4f]" % (mf_b, blo, bhi))

fa = f1_score(y_te, yp_base, average=None, labels=range(10), zero_division=0)
fb_ = f1_score(y_te, yp_rej, average=None, labels=range(10), zero_division=0)
print("\nper-class F1 (baseline -> reject, both @0.5):")
for i, n in enumerate(NAMES):
    print("  %-16s %.4f -> %.4f  (%+.4f)" % (n, fa[i], fb_[i], fb_[i] - fa[i]))

# the recall price of the reject path, measured not assumed
rec_a = recall_score(y_te, yp_base, average=None, labels=range(10), zero_division=0)
rec_b = recall_score(y_te, yp_rej, average=None, labels=range(10), zero_division=0)
print("\nattack recall paid (baseline -> reject):")
for i in range(1, 10):
    print("  %-16s R %.4f -> %.4f  (%+.4f)" % (NAMES[i], rec_a[i], rec_b[i], rec_b[i] - rec_a[i]))

paired delta (reject - baseline) @t=0.5: +0.0242  95% CI [+0.0142, +0.0345]  p ~ 0.000
reject marginal: 0.6843 [0.6674, 0.7002]

per-class F1 (baseline -> reject, both @0.5):
  Benign           0.9915 -> 0.9953  (+0.0038)
  Analysis         0.1693 -> 0.1395  (-0.0298)
  Backdoor         0.6711 -> 0.6623  (-0.0087)
  DoS              0.5680 -> 0.6010  (+0.0330)
  Exploits         0.7927 -> 0.8203  (+0.0276)
  Fuzzers          0.5977 -> 0.6220  (+0.0243)
  Generic          0.8837 -> 0.8772  (-0.0065)
  Reconnaissance   0.8165 -> 0.8246  (+0.0081)
  Shellcode        0.5152 -> 0.7205  (+0.2053)
  Worms            0.5952 -> 0.5800  (-0.0152)

attack recall paid (baseline -> reject):
  Analysis         R 0.4805 -> 0.3117  (-0.1688)
  Backdoor         R 0.5667 -> 0.5667  (+0.0000)
  DoS              R 0.4842 -> 0.5554  (+0.0713)
  Exploits         R 0.9065 -> 0.8632  (-0.0433)
  Fuzzers          R 0.9431 -> 0.5763  (-0.3667)
  Generic          R 0.8877 -> 0.8821  (-0.0056)
  Reconnaissance   

## 7. v7.1b — refit `W` under the END-TO-END objective

Section 4 fitted `W` to maximise **Stage-B 10-class** macro-F1, where the reject class is 1/10 of
the objective. End-to-end that is mis-calibrated: a correct reject barely moves Benign F1
(support 505k) while a swallowed attack costs its family's recall directly. Here the same
coordinate ascent is re-run scoring the **end-to-end** macro-F1 over the COMB half
(reject → Benign; gate-negative rows enter as fixed background). Zero new parameters.

Script twin: `results/v71_w_e2e/` (same numbers; test there: **0.6874 / 0.9866**, paired
**+0.0032 [+0.0019, +0.0047], p<0.001** vs v7.1). These cells reuse the kernel state from
sections 3–6 (`OOF10`, `ARM_R`, `FINAL10`, `p_test`, `yp_rej`), so run the notebook top-to-bottom
first (caches make that fast).

In [13]:
# e2e objective over the COMB half: gate-positives vary with W, the rest is fixed background
gpos_comb = comb_idx[p_gate[comb_idx] > 0.5]
yt_gpos = y_arr[gpos_comb]
P_GPOS = [OOF10[m][gpos_comb] for m in range(M)]
neg_comb = comb_idx[p_gate[comb_idx] <= 0.5]
BG = np.zeros((10, 10), np.int64)
for c, cnt in zip(*np.unique(y_arr[neg_comb], return_counts=True)):
    BG[c, 0] = cnt
print("COMB gate-positives %d | background rows %d" % (len(gpos_comb), len(neg_comb)))

def macro_f1_from_cm(cm):
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    return (2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)).mean()

def e2e_of_W(W):
    yp = LAB10[fuse_static(P_GPOS, W)]
    cm = np.bincount(yt_gpos * 10 + yp, minlength=100).reshape(10, 10) + BG
    return macro_f1_from_cm(cm)

def fit_W_e2e(W0):
    W = W0.copy(); best = e2e_of_W(W)
    for _ in range(3):
        for c in range(NCR):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s_ = e2e_of_W(W)
                    if s_ > best + 1e-7: best, cur = s_, gg
                W[m_, c] = cur
    return W, best

W_sb = ARM_R["W"]
t0 = time.time()
W_e2e_u, e_u = fit_W_e2e(np.ones((M, NCR)) / M)
W_e2e_w, e_w = fit_W_e2e(W_sb)
W_e2e = W_e2e_w if e_w >= e_u else W_e2e_u
print("fitted in %.0fs -- COMB e2e mF1: W_stageB %.4f | uniform %.4f | W_e2e %.4f (init %s)"
      % (time.time() - t0, e2e_of_W(W_sb), e2e_of_W(np.ones((M, NCR)) / M),
         max(e_u, e_w), "warm" if e_w >= e_u else "uniform"))
print("reject column unchanged:", np.allclose(W_sb[:, IREJ], W_e2e[:, IREJ]),
      "-- the ascent rebalances the FAMILY columns, not the reject dial")

COMB gate-positives 49128 | background rows 993593
fitted in 32s -- COMB e2e mF1: W_stageB 0.6875 | uniform 0.6657 | W_e2e 0.6906 (init warm)
reject column unchanged: True -- the ascent rebalances the FAMILY columns, not the reject dial


In [14]:
# SELV (t=0.5): W_e2e variants vs the section-5 winner as reference
P, idx, gmask = selv_eval(ARM_R, 0.5)
Wd30 = comp_ola(ARM_R, idx, 30)
CAND = {"STATIC W_e2e": fuse_static(P, W_e2e),
        "HYBRID W_e2e x OLA-cw k=30 p=1": fuse_dyn(P, Wd30, p=1, Wclass=W_e2e),
        "HYBRID W_e2e x OLA-cw k=30 p=3": fuse_dyn(P, Wd30, p=3, Wclass=W_e2e),
        "v7.1 ref: HYBRID W_sb x OLA-cw k=30 p=3": fuse_dyn(P, Wd30, p=3, Wclass=W_sb)}
tab_b = pd.DataFrame([score_e2e(ARM_R, fg, y_arr[selv_idx], gmask, v) for v, fg in CAND.items()]
                     ).sort_values("e2e_mF1", ascending=False).reset_index(drop=True)
print(tab_b.to_string(index=False))
BEST_B = tab_b[~tab_b.variant.str.startswith("v7.1 ref")].iloc[0]["variant"]
print("\nSELV winner among W_e2e variants:", BEST_B)
del P, idx, Wd30; gc.collect()

                                variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
         HYBRID W_e2e x OLA-cw k=30 p=3   0.6717  0.6996  0.6673   0.9859
         HYBRID W_e2e x OLA-cw k=30 p=1   0.6699  0.6987  0.6655   0.9859
v7.1 ref: HYBRID W_sb x OLA-cw k=30 p=3   0.6695  0.7046  0.6560   0.9870
                           STATIC W_e2e   0.6689  0.6968  0.6651   0.9859

SELV winner among W_e2e variants: HYBRID W_e2e x OLA-cw k=30 p=3


77

In [15]:
# TEST once (reuses the section-6 refit pool FINAL10 and gate probabilities p_test)
gmask_test = p_test > 0.5
gpos_t = np.where(gmask_test)[0]
P_t = [m.predict_proba(X_test.iloc[gpos_t]).astype(np.float32) for m in FINAL10]
_, idx_t = ARM_R["NN"].kneighbors(prof(P_t))
Wd30_t = comp_ola(ARM_R, idx_t, 30)
CAND_T = {"STATIC W_e2e": fuse_static(P_t, W_e2e),
          "HYBRID W_e2e x OLA-cw k=30 p=1": fuse_dyn(P_t, Wd30_t, p=1, Wclass=W_e2e),
          "HYBRID W_e2e x OLA-cw k=30 p=3": fuse_dyn(P_t, Wd30_t, p=3, Wclass=W_e2e)}
tab_t = pd.DataFrame([score_e2e(ARM_R, fg, y_te, gmask_test, v) for v, fg in CAND_T.items()]
                     ).sort_values("e2e_mF1", ascending=False).reset_index(drop=True)
print(tab_t.to_string(index=False))

yp_b = np.zeros(len(y_te), int); yp_b[gmask_test] = ARM_R["LAB"][CAND_T[BEST_B]]
mf_new = f1_score(y_te, yp_b, average="macro", zero_division=0)
acc_new = float((yp_b == y_te).mean())
print("\nv7.1b '%s': test macro-F1 %.4f | accuracy %.4f" % (BEST_B, mf_new, acc_new))

# paired bootstrap vs the section-6 v7.1 predictions (yp_rej)
code_a = y_te * 10 + yp_rej
code_b = y_te * 10 + yp_b
rng = np.random.RandomState(RS)
cls_i = [np.where(y_te == c)[0] for c in range(10)]
diffs = np.empty(2000)
for it in range(2000):
    bidx = np.concatenate([ci[rng.randint(0, len(ci), len(ci))] for ci in cls_i])
    cma = np.bincount(code_a[bidx], minlength=100).reshape(10, 10)
    cmb = np.bincount(code_b[bidx], minlength=100).reshape(10, 10)
    diffs[it] = macro_f1_from_cm(cmb) - macro_f1_from_cm(cma)
lo, hi = np.percentile(diffs, [2.5, 97.5])
p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
print("paired delta (W_e2e - W_sb): %+.4f  95%% CI [%+.4f, %+.4f]  p ~ %.3f"
      % (mf_new - f1_score(y_te, yp_rej, average="macro", zero_division=0), lo, hi, min(1.0, p_two)))

fa = f1_score(y_te, yp_rej, average=None, labels=range(10), zero_division=0)
fb2 = f1_score(y_te, yp_b, average=None, labels=range(10), zero_division=0)
ra = recall_score(y_te, yp_rej, average=None, labels=range(10), zero_division=0)
rb2 = recall_score(y_te, yp_b, average=None, labels=range(10), zero_division=0)
print("\nper-class F1 (W_sb -> W_e2e) and recall:")
for i, n in enumerate(NAMES):
    print("  %-16s F1 %.4f -> %.4f (%+.4f) | R %.4f -> %.4f"
          % (n, fa[i], fb2[i], fb2[i] - fa[i], ra[i], rb2[i]))
print("\nfull-release ladder: 0.6601 (v7) -> %.4f (reject) -> %.4f (e2e W)  --  quote 0.687 [0.671, 0.704]"
      % (f1_score(y_te, yp_rej, average="macro", zero_division=0), mf_new))

                       variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
HYBRID W_e2e x OLA-cw k=30 p=3   0.6874  0.7111  0.6874   0.9866
                  STATIC W_e2e   0.6863  0.7137  0.6849   0.9866
HYBRID W_e2e x OLA-cw k=30 p=1   0.6856  0.7116  0.6853   0.9866

v7.1b 'HYBRID W_e2e x OLA-cw k=30 p=3': test macro-F1 0.6874 | accuracy 0.9866
paired delta (W_e2e - W_sb): +0.0032  95% CI [+0.0019, +0.0047]  p ~ 0.000

per-class F1 (W_sb -> W_e2e) and recall:
  Benign           F1 0.9953 -> 0.9949 (-0.0004) | R 0.9958 -> 0.9935
  Analysis         F1 0.1395 -> 0.1395 (+0.0000) | R 0.3117 -> 0.3117
  Backdoor         F1 0.6623 -> 0.6667 (+0.0043) | R 0.5667 -> 0.5667
  DoS              F1 0.6010 -> 0.6025 (+0.0015) | R 0.5554 -> 0.5554
  Exploits         F1 0.8203 -> 0.8247 (+0.0044) | R 0.8632 -> 0.8464
  Fuzzers          F1 0.6220 -> 0.6413 (+0.0193) | R 0.5763 -> 0.7310
  Generic          F1 0.8772 -> 0.8780 (+0.0008) | R 0.8821 -> 0.8765
  Reconnaissance   F1 0.8246 -> 0.8262 (+0.0016) | R 